In [1]:
from pathlib import Path
import importlib.util
import inspect
import sys

def load_module(module_name, filename_candidates):
    search_roots = [
        Path.cwd(),
        Path.cwd() / "python_version",
        Path.cwd().parent,
    ]
    for root in search_roots:
        for name in filename_candidates:
            path = root / name
            if path.exists():
                spec = importlib.util.spec_from_file_location(module_name, path)
                module = importlib.util.module_from_spec(spec)
                sys.modules[module_name] = module
                spec.loader.exec_module(module)
                print(f"Loaded {module_name} from: {path}")
                return module
    raise FileNotFoundError(f"Could not find any of: {filename_candidates}")

gillespie = load_module("gillespie", ["gillespie.py", "gillespie(7).py"])
prc1 = load_module("prc1", ["prc1.py", "prc1(7).py"])
prc1_state = load_module("prc1_state", ["prc1_state.py", "prc1_state(7).py"])
run_gillespie = load_module("run_gillespie", ["run_gillespie.py", "run_gillespie(6).py"])
smc_abc = load_module("smc_abc", ["smc_abc.py", "smc-abc.py", "smc-abc(6).py"])

sig = inspect.signature(run_gillespie.run_gillespie_prc1_on_grid)
required_flags = {
    "enable_initial_attach_cooperativity",
    "enable_second_head_attach_cooperativity",
    "enable_second_head_detach_cooperativity",
}
missing = sorted(required_flags - set(sig.parameters))
if missing:
    raise RuntimeError(
        "Your loaded run_gillespie module does not have the new cooperativity flags. "
        f"Missing: {missing}. Make sure you are loading the patched files."
    )

print("Patched cooperativity flags detected successfully.")

Loaded gillespie from: C:\Users\Khang\Desktop\abchandbook\prc-recruitment-bayesian-analysis\python_version\gillespie.py
Loaded prc1 from: C:\Users\Khang\Desktop\abchandbook\prc-recruitment-bayesian-analysis\python_version\prc1.py
Loaded prc1_state from: C:\Users\Khang\Desktop\abchandbook\prc-recruitment-bayesian-analysis\python_version\prc1_state.py
Loaded run_gillespie from: C:\Users\Khang\Desktop\abchandbook\prc-recruitment-bayesian-analysis\python_version\run_gillespie.py
Loaded smc_abc from: C:\Users\Khang\Desktop\abchandbook\prc-recruitment-bayesian-analysis\python_version\smc-abc.py
Patched cooperativity flags detected successfully.


In [2]:
import numpy as np

# fixed model parameters
INIT_BIND = 0.002
K_DET_1 = 0.1
K12_FIXED = 5.88
MAX_STEPS = 200_000

# observation grid
times_obs = np.arange(0.0, 68.0 + 4.0, 4.0)

# thermal scale and fixed cooperativity
k_B_T = 4.1
EPS_FIXED = 2.3 * k_B_T

# true k21 used for synthetic data
K21_TRUE = 0.1

print("times_obs =", times_obs)
print("EPS_FIXED =", EPS_FIXED)
print("K12_FIXED =", K12_FIXED)
print("K21_TRUE =", K21_TRUE)

times_obs = [ 0.  4.  8. 12. 16. 20. 24. 28. 32. 36. 40. 44. 48. 52. 56. 60. 64. 68.]
EPS_FIXED = 9.429999999999998
K12_FIXED = 5.88
K21_TRUE = 0.1


## Define a one-parameter simulator

Here `theta[0]` is

`k21 = base_double_detachment_rate`

Everything else is fixed, hopping is disabled, and cooperativity is set to:

- `enable_initial_attach_cooperativity=False`
- `enable_second_head_attach_cooperativity=False`
- `enable_second_head_detach_cooperativity=True`

In [3]:
def simulate_one_k21_detach_only(theta, times_obs, seed):
    """
    theta[0] = k21 = base_double_detachment_rate
    All other parameters are fixed.
    Hopping is disabled.
    Cooperativity acts only in second-head detachment.
    """
    import numpy as np
    np.random.seed(int(seed))

    return run_gillespie.run_gillespie_prc1_on_grid(
        initial_binding_rate_per_site=INIT_BIND,
        singly_bound_detachment_rate=K_DET_1,
        base_double_attachment_rate=K12_FIXED,
        base_double_detachment_rate=float(theta[0]),
        times_obs=times_obs,
        cooperativity_energy=EPS_FIXED,
        enable_initial_attach_cooperativity=False,
        enable_second_head_attach_cooperativity=False,
        enable_second_head_detach_cooperativity=True,
        enable_hopping=False,
        max_steps=MAX_STEPS,
    )

In [4]:
import time

theta_test = np.array([K21_TRUE], dtype=float)

t0 = time.perf_counter()
test_path = simulate_one_k21_detach_only(theta_test, times_obs, seed=123)
t1 = time.perf_counter()

one_sim_seconds = t1 - t0

print("One simulation path:")
print(test_path)
print("Shape:", test_path.shape)
print(f"Time for one simulation: {one_sim_seconds:.6f} seconds")

One simulation path:
[  0.  14.  21.  32.  41.  47.  54.  65.  70.  72.  79.  90. 103. 111.
 114. 121. 128. 130.]
Shape: (18,)
Time for one simulation: 10.285751 seconds


In [5]:
smc_abc.simulate_one = simulate_one_k21_detach_only
print("Patched smc_abc.simulate_one successfully.")
print("joblib available:", getattr(smc_abc, "_JOBLIB_AVAILABLE", None))

Patched smc_abc.simulate_one successfully.
joblib available: True


In [6]:
n_obs_reps = 10

y_obs = np.array([
    run_gillespie.run_gillespie_prc1_on_grid(
        initial_binding_rate_per_site=INIT_BIND,
        singly_bound_detachment_rate=K_DET_1,
        base_double_attachment_rate=K12_FIXED,
        base_double_detachment_rate=K21_TRUE,
        times_obs=times_obs,
        cooperativity_energy=EPS_FIXED,
        enable_initial_attach_cooperativity=False,
        enable_second_head_attach_cooperativity=False,
        enable_second_head_detach_cooperativity=True,
        enable_hopping=False,
        max_steps=MAX_STEPS,
    )
    for _ in range(n_obs_reps)
], dtype=float)

print("y_obs shape:", y_obs.shape)
print(y_obs)

y_obs shape: (10, 18)
[[  0.   8.  20.  30.  38.  45.  53.  63.  77.  79.  85.  93. 101. 112.
  118. 121. 120. 123.]
 [  0.   2.   5.  13.  22.  28.  34.  39.  47.  58.  61.  66.  72.  75.
   81.  87.  92. 100.]
 [  0.  11.  22.  27.  35.  43.  48.  57.  64.  73.  81.  84.  93.  98.
  109. 112. 124. 127.]
 [  0.  12.  21.  28.  42.  51.  58.  67.  72.  80.  88.  95.  97. 106.
  109. 112. 114. 121.]
 [  0.  11.  19.  25.  36.  44.  48.  55.  62.  71.  81.  87.  97. 101.
  107. 117. 122. 131.]
 [  0.   7.  10.  24.  36.  51.  56.  69.  77.  89.  90. 106. 113. 118.
  130. 135. 135. 135.]
 [  0.  11.  17.  20.  27.  37.  46.  55.  60.  69.  75.  80.  93.  95.
  103. 109. 117. 122.]
 [  0.   6.  12.  22.  34.  43.  54.  61.  75.  80.  90.  96. 100. 104.
  106. 109. 117. 118.]
 [  0.   7.  18.  27.  37.  43.  50.  59.  65.  73.  86.  95.  98. 100.
  102. 106. 116. 121.]
 [  0.   6.  21.  28.  38.  52.  56.  60.  68.  80.  85.  95.  99. 104.
  105. 108. 113. 121.]]


In [7]:
K21_center = 0.3   # intentionally inaccurate prior center
phi_mu = np.array([np.log(K21_center)], dtype=float)
phi_sd = np.array([0.5], dtype=float)

print("true k21 =", K21_TRUE)
print("prior center =", K21_center)
print("phi_mu =", phi_mu)
print("phi_sd =", phi_sd)

true k21 = 0.1
prior center = 0.3
phi_mu = [-1.2039728]
phi_sd = [0.5]


In [ ]:
res = smc_abc.smc_abc_prc1(
    times_obs=times_obs,
    y_obs=y_obs,
    phi_mu=phi_mu,
    phi_sd=phi_sd,
    P=30,
    G=3,
    pool=80,
    n_reps=3,
    eps_quantile=60.0,
    cov_scale=2.0,
    seed=1,
    n_jobs=-1,
    batch_factor=4,
)

print("Finished SMC-ABC.")
print("eps history:", res.eps_history)

[SMC-ABC] Starting pilot run
[SMC-ABC] Time: 2026-04-17 16:28:32


In [ ]:
posterior_k21 = np.exp(res.particles_phi[:, 0])
posterior_w = res.weights

print("posterior k21 particles:")
print(posterior_k21)
print("posterior weights:")
print(posterior_w)

In [ ]:
post_mean = np.sum(posterior_w * posterior_k21)

order = np.argsort(posterior_k21)
k21_sorted = posterior_k21[order]
w_sorted = posterior_w[order]
cdf = np.cumsum(w_sorted)
post_median = k21_sorted[np.searchsorted(cdf, 0.5)]

print("True k21:", K21_TRUE)
print("Posterior mean:", post_mean)
print("Posterior median:", post_median)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.hist(posterior_k21, bins=10, weights=posterior_w)
plt.axvline(K21_center, linestyle=":", label="prior center")
plt.axvline(K21_TRUE, linestyle="--", label="true value")
plt.xlabel("k21")
plt.ylabel("posterior weight")
plt.title("Posterior for k21 with detachment-only cooperativity")
plt.legend()
plt.show()

In [ ]:
from scipy.stats import gaussian_kde

posterior_samples = np.asarray(posterior_k21, dtype=float)
posterior_w = np.asarray(posterior_w, dtype=float)

phi0 = float(phi_mu[0])
sd0 = float(phi_sd[0])

def prior_density(x):
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)
    mask = x > 0
    z = (np.log(x[mask]) - phi0) / sd0
    out[mask] = np.exp(-0.5 * z * z) / (x[mask] * sd0 * np.sqrt(2 * np.pi))
    return out

kde_post = gaussian_kde(posterior_samples, weights=posterior_w)

xmin = min(np.min(posterior_samples), K21_TRUE, K21_center) * 0.6
xmax = max(np.max(posterior_samples), K21_TRUE, K21_center) * 1.4
xmin = max(xmin, 1e-8)

xgrid = np.linspace(xmin, xmax, 500)

prior_y = prior_density(xgrid)
post_y = kde_post(xgrid)

plt.figure(figsize=(7, 4.5))
plt.plot(xgrid, prior_y, label="Prior")
plt.plot(xgrid, post_y, label="Posterior")
plt.axvline(K21_center, linestyle=":", label="Prior center")
plt.axvline(K21_TRUE, linestyle="--", label="True value")
plt.xlabel("k21")
plt.ylabel("Density")
plt.title("Prior vs posterior for k21 (inaccurate prior)")
plt.legend()
plt.show()